<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/process/LNG_Liquefaction_Processes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LNG process efficiency: SMR, C3MR, DMR, and nitrogen expansion

This notebook constructs and runs four independent, closed-loop LNG process models with
NeqSim's `LNGProcessBuilder`. Every route starts from a fresh copy of the same synthetic,
pretreated natural-gas feed and ends at the same LNG product-flash pressure.

**Learning outcomes**

1. configure SMR, C3MR, DMR, and nitrogen-expander process models independently;
2. inspect the explicit compressors, coolers, exchangers, valves, separators, and recycles;
3. calculate LNG yield, net shaft power, specific energy, refrigeration COP, and exchanger metrics;
4. close total mass and natural-gas enthalpy balances;
5. compare simulated specific energy with four literature points without treating them as
   universal acceptance limits; and
6. test whether the SMR result is reasonably independent of exchanger zone count.

The models are for route screening, teaching, optimization studies, and API regression—not
licensor design, FEED verification, or an equipment guarantee.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys
import zipfile
from pathlib import Path

REQUIRED_NEQSIM_COMMIT = "00200d037fb55f52ec4a98d52e5411744443cda2"
SOURCE_DIRECTORY = Path("/tmp/neqsim-lng-source")
BUILD_JAR = None

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "neqsim",
        "pandas",
        "matplotlib",
    ]
)

package_spec = importlib.util.find_spec("neqsim")
package_directory = Path(package_spec.submodule_search_locations[0])
installed_jars = sorted((package_directory / "lib").glob("*.jar"))

builder_path = "neqsim/process/lng/LNGProcessBuilder.class"
release_has_builder = any(
    builder_path in zipfile.ZipFile(jar_path).namelist()
    for jar_path in installed_jars
)

if release_has_builder:
    from neqsim import jneqsim

    NEQSIM_BUILD = importlib.metadata.version("neqsim")
else:
    if not SOURCE_DIRECTORY.exists():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "https://github.com/equinor/neqsim.git",
                str(SOURCE_DIRECTORY),
            ]
        )
        subprocess.check_call(
            [
                "git",
                "-C",
                str(SOURCE_DIRECTORY),
                "fetch",
                "--depth",
                "1",
                "origin",
                REQUIRED_NEQSIM_COMMIT,
            ]
        )
        subprocess.check_call(
            [
                "git",
                "-C",
                str(SOURCE_DIRECTORY),
                "checkout",
                "--detach",
                REQUIRED_NEQSIM_COMMIT,
            ]
        )

    subprocess.check_call(
        [
            str(SOURCE_DIRECTORY / "mvnw"),
            "-q",
            "-DskipTests",
            "-Dspotless.skip=true",
            "-Dcheckstyle.skip=true",
            "-Dpmd.skip=true",
            "-Dspotbugs.skip=true",
            "package",
        ],
        cwd=SOURCE_DIRECTORY,
    )

    jar_candidates = [
        jar_path
        for jar_path in (SOURCE_DIRECTORY / "target").glob("*.jar")
        if "sources" not in jar_path.name
        and "javadoc" not in jar_path.name
        and not jar_path.name.startswith("original-")
    ]
    BUILD_JAR = max(jar_candidates, key=lambda jar_path: jar_path.stat().st_size)

    import jpype

    if not jpype.isJVMStarted():
        jpype.startJVM(classpath=[str(BUILD_JAR)])

    jneqsim = jpype.JPackage("neqsim")
    NEQSIM_BUILD = f"source commit {REQUIRED_NEQSIM_COMMIT[:12]}"

print(f"NeqSim build: {NEQSIM_BUILD}")
print(f"Python version: {sys.version.split()[0]}")

if BUILD_JAR is not None:
    print(f"Built JAR: {BUILD_JAR.name}")


## 1. Reproducible model basis

The feed represents gas **after** acid-gas removal, dehydration, mercury removal, and
heavy-hydrocarbon control. These pretreatment systems are outside the notebook boundary.

| Input | Common value |
|---|---:|
| Feed flow | 20,000 kg/h |
| Feed temperature | 25 °C |
| Feed pressure | 60 bara |
| LNG flash pressure | 1.20 bara |
| Natural-gas temperature before letdown | -155 °C |
| Minimum exchanger approach setting | 3 °C |
| Compressor isentropic efficiency | 0.78 |
| Expander isentropic efficiency | 0.82 |
| Thermodynamic model | SRK, classic mixing rule |

The 20,000 kg/h rate matches the scale reported for the Pereira et al. comparison, but the
feed composition and full study basis are not asserted to match. Specific energy can only be
interpreted as like-for-like when feed, ambient sink, product definition, pressure losses,
driver efficiencies, and flowsheet detail also match.

The synthetic feed composition obeys:

$$
\sum_i z_i = 1
$$

where $z_i$ is overall mole fraction in mol/mol.


In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
LNGProcessBuilder = jneqsim.process.lng.LNGProcessBuilder
LNGProcessCycle = jneqsim.process.lng.LNGProcessCycle
LNGProcessModel = jneqsim.process.lng.LNGProcessModel

FEED_FLOW_KG_PER_H = 20_000.0
FEED_TEMPERATURE_C = 25.0
FEED_PRESSURE_BARA = 60.0
PRODUCT_PRESSURE_BARA = 1.20
TARGET_LIQUEFACTION_TEMPERATURE_C = -155.0
MINIMUM_APPROACH_C = 3.0
NUMBER_OF_ZONES = 12
COMPRESSOR_EFFICIENCY = 0.78
EXPANDER_EFFICIENCY = 0.82
OPERATING_HOURS_PER_YEAR = 8_000.0

FEED_COMPOSITION = {
    "nitrogen": 0.005,
    "methane": 0.890,
    "ethane": 0.065,
    "propane": 0.025,
    "i-butane": 0.007,
    "n-butane": 0.008,
}

composition_sum = sum(FEED_COMPOSITION.values())
assert np.isclose(composition_sum, 1.0, atol=1.0e-12)

composition_table = pd.DataFrame(
    {
        "component": list(FEED_COMPOSITION),
        "mole fraction [mol/mol]": list(FEED_COMPOSITION.values()),
    }
)

display(composition_table)
print(f"Composition sum [mol/mol]: {composition_sum:.6f}")


## 2. Feed fluid and shared result calculations

A new SRK fluid is created for every route. `setFeedFluid` clones it, so none of the
four process models shares mutable thermodynamic state.

NeqSim reports the principal route-comparison metric directly:

$$
e_{\mathrm{LNG}} =
\frac{\dot W_{\mathrm{compressors}} - \lvert\dot W_{\mathrm{expanders}}\rvert}
{\dot m_{\mathrm{LNG}}}
$$

Here $e_{\mathrm{LNG}}$ is in kWh/kg LNG when power is in kW and LNG flow is in kg/h.

The notebook also calculates the heat removed from the natural gas:

$$
\dot Q_{\mathrm{NG}} =
\dot H_{\mathrm{feed}}
- \dot H_{\mathrm{LNG}}
- \dot H_{\mathrm{flash\ gas}}
$$

and the screening refrigeration coefficient of performance:

$$
\mathrm{COP}_{\mathrm{NG}} =
\frac{\dot Q_{\mathrm{NG}}}{\dot W_{\mathrm{net}}}
$$

This COP is a stated process-boundary metric, not a complete plant second-law efficiency.


In [ ]:
def make_feed_fluid():
    fluid = SystemSrkEos(
        FEED_TEMPERATURE_C + 273.15,
        FEED_PRESSURE_BARA,
    )

    for component_name, mole_fraction in FEED_COMPOSITION.items():
        fluid.addComponent(component_name, float(mole_fraction))

    fluid.setMixingRule("classic")
    return fluid


def enthalpy_rate_kw(stream):
    mass_rate_kg_per_h = stream.getFlowRate("kg/hr")
    specific_enthalpy_j_per_kg = stream.getFluid().getEnthalpy("J/kg")
    return mass_rate_kg_per_h * specific_enthalpy_j_per_kg / 3.6e6


def result_record(model, result, display_name):
    feed_stream = model.getFeedStream()
    lng_stream = model.getOutputStream(LNGProcessModel.LNG_OUTPUT)
    flash_gas_stream = model.getOutputStream(
        LNGProcessModel.FLASH_GAS_OUTPUT
    )

    natural_gas_duty_kw = (
        enthalpy_rate_kw(feed_stream)
        - enthalpy_rate_kw(lng_stream)
        - enthalpy_rate_kw(flash_gas_stream)
    )
    mass_residual_kg_per_h = (
        feed_stream.getFlowRate("kg/hr")
        - lng_stream.getFlowRate("kg/hr")
        - flash_gas_stream.getFlowRate("kg/hr")
    )
    net_power_kw = result.getNetPowerKW()
    refrigeration_cop = natural_gas_duty_kw / net_power_kw

    return {
        "cycle": display_name,
        "feed [kg/h]": result.getFeedMassFlowKgPerHour(),
        "LNG [kg/h]": result.getLNGMassFlowKgPerHour(),
        "LNG yield [kg/kg]": result.getLNGYield(),
        "capacity [MTPA]": result.getCapacityMTPA(),
        "compressor power [kW]": result.getCompressorPowerKW(),
        "expander recovery [kW]": result.getExpanderRecoveredPowerKW(),
        "net power [kW]": net_power_kw,
        "specific energy [kWh/kg LNG]": (
            result.getSpecificEnergyKWhPerKgLNG()
        ),
        "natural-gas duty [kW]": natural_gas_duty_kw,
        "refrigeration COP [-]": refrigeration_cop,
        "product temperature [°C]": result.getProductTemperatureC(),
        "product pressure [bara]": result.getProductPressureBara(),
        "product density [kg/m³]": result.getProductDensityKgPerM3(),
        "minimum approach [°C]": (
            result.getMinimumInternalTemperatureApproachC()
        ),
        "exchanger exergy destruction [kW]": (
            result.getExchangerExergyDestructionKW()
        ),
        "mass residual [kg/h]": mass_residual_kg_per_h,
        "runtime [ms]": result.getRunTimeMilliseconds(),
    }


## 3. Single mixed refrigerant (SMR)

The SMR template contains one mixed-refrigerant loop with a suction scrubber, two compression
stages, intercooling and aftercooling, JT expansion, a multi-stream main cryogenic exchanger,
and a recycle tear. The product section adds high-pressure natural-gas letdown and an
equilibrium flash separator.


In [ ]:
smr_model = (
    LNGProcessBuilder()
    .setName("SMR from-scratch comparison")
    .setCycle(LNGProcessCycle.SMR)
    .setFeedFluid(make_feed_fluid())
    .setFeedFlowRate(FEED_FLOW_KG_PER_H)
    .setFeedTemperature(FEED_TEMPERATURE_C)
    .setFeedPressure(FEED_PRESSURE_BARA)
    .setProductPressure(PRODUCT_PRESSURE_BARA)
    .setTargetLiquefactionTemperature(
        TARGET_LIQUEFACTION_TEMPERATURE_C
    )
    .setMinimumApproach(MINIMUM_APPROACH_C)
    .setNumberOfZones(NUMBER_OF_ZONES)
    .setAdaptiveRefinement(True)
    .setUseFlashWarmStart(True)
    .setCompressorEfficiency(COMPRESSOR_EFFICIENCY)
    .build()
)

smr_result = smr_model.run()
smr_record = result_record(smr_model, smr_result, "SMR")
display(pd.DataFrame([smr_record]))


## 4. Propane-precooled mixed refrigerant (C3MR)

This route adds a propane loop ahead of the mixed-refrigerant main exchanger. The current
screening template uses one equivalent propane evaporation level. A baseload design normally
uses three or four propane pressure levels plus project-specific refrigerant inventory,
compressor maps, and exchanger geometry.


In [ ]:
c3mr_model = (
    LNGProcessBuilder()
    .setName("C3MR from-scratch comparison")
    .setCycle(LNGProcessCycle.C3MR)
    .setFeedFluid(make_feed_fluid())
    .setFeedFlowRate(FEED_FLOW_KG_PER_H)
    .setFeedTemperature(FEED_TEMPERATURE_C)
    .setFeedPressure(FEED_PRESSURE_BARA)
    .setProductPressure(PRODUCT_PRESSURE_BARA)
    .setTargetLiquefactionTemperature(
        TARGET_LIQUEFACTION_TEMPERATURE_C
    )
    .setMinimumApproach(MINIMUM_APPROACH_C)
    .setNumberOfZones(NUMBER_OF_ZONES)
    .setAdaptiveRefinement(True)
    .setUseFlashWarmStart(True)
    .setCompressorEfficiency(COMPRESSOR_EFFICIENCY)
    .build()
)

c3mr_result = c3mr_model.run()
c3mr_record = result_record(c3mr_model, c3mr_result, "C3MR")
display(pd.DataFrame([c3mr_record]))


## 5. Dual mixed refrigerant (DMR)

The DMR template has a warm mixed-refrigerant loop for precooling and a cold
mixed-refrigerant loop for liquefaction and subcooling. Both loops have explicit two-stage
compression, cooling, expansion, exchanger passages, and recycle tears.


In [ ]:
dmr_model = (
    LNGProcessBuilder()
    .setName("DMR from-scratch comparison")
    .setCycle(LNGProcessCycle.DMR)
    .setFeedFluid(make_feed_fluid())
    .setFeedFlowRate(FEED_FLOW_KG_PER_H)
    .setFeedTemperature(FEED_TEMPERATURE_C)
    .setFeedPressure(FEED_PRESSURE_BARA)
    .setProductPressure(PRODUCT_PRESSURE_BARA)
    .setTargetLiquefactionTemperature(
        TARGET_LIQUEFACTION_TEMPERATURE_C
    )
    .setMinimumApproach(MINIMUM_APPROACH_C)
    .setNumberOfZones(NUMBER_OF_ZONES)
    .setAdaptiveRefinement(True)
    .setUseFlashWarmStart(True)
    .setCompressorEfficiency(COMPRESSOR_EFFICIENCY)
    .build()
)

dmr_result = dmr_model.run()
dmr_record = result_record(dmr_model, dmr_result, "DMR")
display(pd.DataFrame([dmr_record]))


## 6. Closed nitrogen-expander cycle

The NeqSim template is a closed reverse-Brayton nitrogen loop with two-stage compression,
aftercooling, a multi-stream exchanger, turboexpansion, and recycle. The expander power is
credited against compressor power.

The literature point used later is for an optimized **parallel nitrogen expansion** process.
That topology is not identical to this single closed-loop template, so its deviation is
diagnostic rather than a pass/fail comparison.


In [ ]:
nitrogen_model = (
    LNGProcessBuilder()
    .setName("Nitrogen-expander from-scratch comparison")
    .setCycle(LNGProcessCycle.NITROGEN_EXPANDER)
    .setFeedFluid(make_feed_fluid())
    .setFeedFlowRate(FEED_FLOW_KG_PER_H)
    .setFeedTemperature(FEED_TEMPERATURE_C)
    .setFeedPressure(FEED_PRESSURE_BARA)
    .setProductPressure(PRODUCT_PRESSURE_BARA)
    .setTargetLiquefactionTemperature(
        TARGET_LIQUEFACTION_TEMPERATURE_C
    )
    .setMinimumApproach(MINIMUM_APPROACH_C)
    .setNumberOfZones(NUMBER_OF_ZONES)
    .setAdaptiveRefinement(True)
    .setUseFlashWarmStart(True)
    .setCompressorEfficiency(COMPRESSOR_EFFICIENCY)
    .setExpanderEfficiency(EXPANDER_EFFICIENCY)
    .build()
)

nitrogen_result = nitrogen_model.run()
nitrogen_record = result_record(
    nitrogen_model,
    nitrogen_result,
    "Nitrogen expander",
)
display(pd.DataFrame([nitrogen_record]))


## 7. Verify that the models contain complete route equipment

The builder is not a single-duty shortcut. `getEquipment()` exposes every unit operation in
flowsheet order, including suction scrubbers, compressors, intercoolers, aftercoolers,
cryogenic exchangers, expansion devices, separators, and recycle units. The two named product
streams are ordinary NeqSim streams and can be connected to downstream equipment.


In [ ]:
route_models = {
    "SMR": smr_model,
    "C3MR": c3mr_model,
    "DMR": dmr_model,
    "Nitrogen expander": nitrogen_model,
}

equipment_records = []

for cycle_name, model in route_models.items():
    for sequence, equipment in enumerate(model.getEquipment(), start=1):
        equipment_records.append(
            {
                "cycle": cycle_name,
                "sequence": sequence,
                "equipment": str(equipment.getName()),
                "type": str(equipment.getClass().getSimpleName()),
            }
        )

equipment_table = pd.DataFrame(equipment_records)
equipment_summary = (
    equipment_table.groupby(["cycle", "type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(equipment_summary)
display(equipment_table)


## 8. Common-basis efficiency and performance table

The principal ranking metric is net shaft energy per kilogram of liquid LNG. LNG yield is
reported separately so that a low energy number cannot hide an excessive flash-gas loss.
The COP reports useful natural-gas enthalpy removal divided by net shaft input within the
defined model boundary.

Exchanger exergy destruction and MITA are diagnostics. Negative MITA indicates a temperature
cross. Very small positive MITA may be numerically feasible but difficult to realize with a
finite exchanger area.


In [ ]:
route_records = [
    smr_record,
    c3mr_record,
    dmr_record,
    nitrogen_record,
]
results_table = pd.DataFrame(route_records).set_index("cycle")

display(
    results_table[
        [
            "LNG yield [kg/kg]",
            "capacity [MTPA]",
            "net power [kW]",
            "specific energy [kWh/kg LNG]",
            "refrigeration COP [-]",
            "minimum approach [°C]",
            "exchanger exergy destruction [kW]",
        ]
    ].round(5)
)


## 9. Literature comparison points

These are **comparison points, not universal acceptance limits**. Feed composition, ambient
conditions, LNG flash definition, driver efficiency, pressure losses, and flowsheet complexity
must match before interpreting deviations.

| Cycle | Reference specific energy (kWh/kg LNG) | Source |
|---|---:|---|
| SMR | 0.2561 | Pereira et al. (2022) |
| C3MR | 0.2548 | Pereira et al. (2022) |
| DMR | 0.2456 | Pereira et al. (2022) |
| Parallel nitrogen expansion | 0.6180 | He et al. (2019) |

Pereira et al. compared optimized routes for a common 20,000 kg/h feed:
[Energy Conversion and Management 272 (2022) 116364](https://doi.org/10.1016/j.enconman.2022.116364).

The nitrogen point is the optimized parallel nitrogen expansion case reported by He et al.:
[Energy 167 (2019) 1–12](https://doi.org/10.1016/j.energy.2018.10.169).

The deviation is calculated as:

$$
\delta_e =
100\,
\frac{e_{\mathrm{simulated}} - e_{\mathrm{reference}}}
{e_{\mathrm{reference}}}
$$

A separate literature-energy ratio, $100e_{\mathrm{reference}}/e_{\mathrm{simulated}}$,
is shown only as a normalized energy index. It is **not** a thermodynamic efficiency.


In [ ]:
literature_table = pd.DataFrame(
    [
        {
            "cycle": "SMR",
            "reference specific energy [kWh/kg LNG]": 0.2561,
            "source": "Pereira et al. (2022)",
        },
        {
            "cycle": "C3MR",
            "reference specific energy [kWh/kg LNG]": 0.2548,
            "source": "Pereira et al. (2022)",
        },
        {
            "cycle": "DMR",
            "reference specific energy [kWh/kg LNG]": 0.2456,
            "source": "Pereira et al. (2022)",
        },
        {
            "cycle": "Nitrogen expander",
            "reference specific energy [kWh/kg LNG]": 0.6180,
            "source": "He et al. (2019), parallel N₂ expansion",
        },
    ]
).set_index("cycle")

comparison_table = results_table.join(literature_table)
comparison_table["absolute deviation [kWh/kg LNG]"] = (
    comparison_table["specific energy [kWh/kg LNG]"]
    - comparison_table["reference specific energy [kWh/kg LNG]"]
)
comparison_table["relative deviation [%]"] = (
    100.0
    * comparison_table["absolute deviation [kWh/kg LNG]"]
    / comparison_table["reference specific energy [kWh/kg LNG]"]
)
comparison_table["literature energy ratio [%]"] = (
    100.0
    * comparison_table["reference specific energy [kWh/kg LNG]"]
    / comparison_table["specific energy [kWh/kg LNG]"]
)

display(
    comparison_table[
        [
            "specific energy [kWh/kg LNG]",
            "reference specific energy [kWh/kg LNG]",
            "absolute deviation [kWh/kg LNG]",
            "relative deviation [%]",
            "literature energy ratio [%]",
            "source",
        ]
    ].round(4)
)


## 10. Visual comparison and engineering interpretation

The left panel compares simulated and literature specific energy. The right panel separates
gross compressor power, recovered expander power, and net power. Interpret route differences
together with LNG yield, MITA, and the topology notes—not from a single bar.


In [ ]:
cycle_order = list(comparison_table.index)
positions = np.arange(len(cycle_order))
bar_width = 0.36

figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13.0, 5.0),
)

axes[0].bar(
    positions - bar_width / 2.0,
    comparison_table["specific energy [kWh/kg LNG]"],
    width=bar_width,
    label="NeqSim screening model",
    color="#1f77b4",
)
axes[0].bar(
    positions + bar_width / 2.0,
    comparison_table["reference specific energy [kWh/kg LNG]"],
    width=bar_width,
    label="Literature point",
    color="#ff7f0e",
)
axes[0].set_xticks(positions, cycle_order, rotation=20, ha="right")
axes[0].set_ylabel("Specific energy [kWh/kg LNG]")
axes[0].set_title("Specific-energy comparison")
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend()

axes[1].bar(
    positions,
    comparison_table["compressor power [kW]"],
    label="Compressor power",
    color="#d62728",
)
axes[1].bar(
    positions,
    -comparison_table["expander recovery [kW]"],
    label="Expander recovery",
    color="#2ca02c",
)
axes[1].plot(
    positions,
    comparison_table["net power [kW]"],
    color="black",
    marker="o",
    linewidth=2.0,
    label="Net power",
)
axes[1].axhline(0.0, color="black", linewidth=0.8)
axes[1].set_xticks(positions, cycle_order, rotation=20, ha="right")
axes[1].set_ylabel("Shaft power [kW]")
axes[1].set_title("Rotating-equipment power")
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend()

figure.tight_layout()
plt.show()


## 11. Built-in broad benchmark assessment

NeqSim also supplies broad screening envelopes. The checks cover specific energy, atmospheric
LNG temperature, liquid yield, and absence of a negative exchanger approach. Passing means the
result is physically plausible on that broad basis; failing identifies a diagnostic to
investigate. Neither outcome replaces a matched validation case.


In [ ]:
route_results = {
    "SMR": smr_result,
    "C3MR": c3mr_result,
    "DMR": dmr_result,
    "Nitrogen expander": nitrogen_result,
}
assessment_records = []

for cycle_name, result in route_results.items():
    assessment = result.assessBenchmark()
    messages = [str(message) for message in assessment.getMessages()]
    diagnostic = "All broad screening checks passed"

    if messages:
        diagnostic = " | ".join(messages)

    assessment_records.append(
        {
            "cycle": cycle_name,
            "all broad checks pass": assessment.isWithinRange(),
            "energy in envelope": assessment.isEnergyWithinRange(),
            "product temperature in range": (
                assessment.isProductTemperatureWithinRange()
            ),
            "yield in range": assessment.isYieldWithinRange(),
            "temperature approach valid": (
                assessment.isTemperatureApproachValid()
            ),
            "specific-energy deviation [kWh/kg LNG]": (
                assessment.getSpecificEnergyDeviation()
            ),
            "diagnostic": diagnostic,
        }
    )

assessment_table = pd.DataFrame(assessment_records).set_index("cycle")
display(assessment_table)


## 12. SMR exchanger-grid sensitivity

A process result should not be accepted solely because one discretization converges. The next
calculation rebuilds the SMR model at 6, 12, and 18 initial exchanger zones with adaptive
refinement disabled, isolating the effect of the initial grid. The specific-energy and MITA
spread should become small as the grid is refined.

This is a numerical sensitivity check, not refrigerant optimization.


In [ ]:
zone_sensitivity_records = []

for zone_count in [6, 12, 18]:
    zone_model = (
        LNGProcessBuilder()
        .setName(f"SMR {zone_count}-zone sensitivity")
        .setCycle(LNGProcessCycle.SMR)
        .setFeedFluid(make_feed_fluid())
        .setFeedFlowRate(FEED_FLOW_KG_PER_H)
        .setFeedTemperature(FEED_TEMPERATURE_C)
        .setFeedPressure(FEED_PRESSURE_BARA)
        .setProductPressure(PRODUCT_PRESSURE_BARA)
        .setTargetLiquefactionTemperature(
            TARGET_LIQUEFACTION_TEMPERATURE_C
        )
        .setMinimumApproach(MINIMUM_APPROACH_C)
        .setNumberOfZones(zone_count)
        .setAdaptiveRefinement(False)
        .setUseFlashWarmStart(True)
        .setCompressorEfficiency(COMPRESSOR_EFFICIENCY)
        .build()
    )
    zone_result = zone_model.run()
    zone_sensitivity_records.append(
        {
            "initial zones [-]": zone_count,
            "specific energy [kWh/kg LNG]": (
                zone_result.getSpecificEnergyKWhPerKgLNG()
            ),
            "minimum approach [°C]": (
                zone_result.getMinimumInternalTemperatureApproachC()
            ),
            "runtime [ms]": zone_result.getRunTimeMilliseconds(),
        }
    )

zone_sensitivity_table = pd.DataFrame(zone_sensitivity_records)
energy_range = (
    zone_sensitivity_table["specific energy [kWh/kg LNG]"].max()
    - zone_sensitivity_table["specific energy [kWh/kg LNG]"].min()
)
energy_mean = zone_sensitivity_table[
    "specific energy [kWh/kg LNG]"
].mean()
relative_zone_spread_percent = 100.0 * energy_range / energy_mean

display(zone_sensitivity_table.round(6))
print(
    "Specific-energy spread across tested grids [%]: "
    f"{relative_zone_spread_percent:.4f}"
)


## 13. Engineering verification

The assertions below are intentionally physical rather than tuned to reproduce the literature
numbers. They require four finite results, positive LNG production and net power, total mass
closure, a physical LNG yield, correct final pressure, and no negative finite exchanger
approach. The exact four literature points are also guarded against accidental edits.


In [ ]:
expected_references = {
    "SMR": 0.2561,
    "C3MR": 0.2548,
    "DMR": 0.2456,
    "Nitrogen expander": 0.6180,
}
verification_checks = {}

verification_checks["composition closes"] = np.isclose(
    composition_sum,
    1.0,
    atol=1.0e-12,
)
verification_checks["four routes evaluated"] = len(results_table) == 4
verification_checks["literature values retained"] = all(
    np.isclose(
        literature_table.loc[cycle_name][
            "reference specific energy [kWh/kg LNG]"
        ],
        reference_value,
        atol=1.0e-12,
    )
    for cycle_name, reference_value in expected_references.items()
)
verification_checks["finite numeric results"] = np.isfinite(
    results_table.select_dtypes(include="number")
).all().all()
verification_checks["positive LNG flow"] = (
    results_table["LNG [kg/h]"] > 0.0
).all()
verification_checks["physical LNG yield"] = results_table[
    "LNG yield [kg/kg]"
].between(0.0, 1.000001).all()
verification_checks["positive net power"] = (
    results_table["net power [kW]"] > 0.0
).all()
verification_checks["positive specific energy"] = (
    results_table["specific energy [kWh/kg LNG]"] > 0.0
).all()
verification_checks["mass closes"] = (
    results_table["mass residual [kg/h]"].abs() < 1.0e-5
).all()
verification_checks["product pressure matches"] = np.allclose(
    results_table["product pressure [bara]"],
    PRODUCT_PRESSURE_BARA,
    atol=1.0e-8,
)
verification_checks["no finite temperature cross"] = all(
    not math.isfinite(approach_c) or approach_c >= 0.0
    for approach_c in results_table["minimum approach [°C]"]
)
verification_checks["zone study finite"] = np.isfinite(
    zone_sensitivity_table.select_dtypes(include="number")
).all().all()

failed_checks = [
    check_name
    for check_name, passed in verification_checks.items()
    if not passed
]

assert not failed_checks, failed_checks

print(f"Engineering checks passed: {len(verification_checks)}")

for check_name in verification_checks:
    print(f"PASS - {check_name}")


## 14. What the comparison does—and does not—establish

**What is calculated consistently**

- the same pretreated feed composition, flow, temperature, and pressure for all routes;
- the same high-pressure natural-gas target temperature and product flash pressure;
- explicit compressor and expander isentropic efficiencies;
- closed refrigerant recycles and energy-coupled multi-stream exchangers;
- liquid LNG and flash-gas outlet streams;
- net specific energy, LNG yield, COP, MITA, exchanger exergy destruction, and runtime; and
- route-specific literature deviation on a clearly labeled basis.

**Important limitations**

- Refrigerant compositions, circulation rates, and pressure levels are template defaults, not
  optimized for this feed.
- C3MR has one equivalent propane evaporation level rather than a detailed multilevel cascade.
- Exchanger geometry, heat leak, passage pressure drops, compressor maps, driver losses,
  pretreatment, utilities, boil-off-gas handling, controls, availability, and economics are
  outside the comparison.
- SRK with the classic mixing rule is a screening choice; project work should validate the
  thermodynamic model against representative VLE, density, enthalpy, and LNG property data.
- The nitrogen literature case is not topologically identical to the NeqSim closed
  nitrogen-expander template.
- A matched independent simulation or plant dataset is required before judging model accuracy.

For optimization, converge a base case first, then vary refrigerant composition, inventory,
pressure levels, exchanger approach, and equipment efficiencies within explicit constraints.


## 15. Summary and next exercises

This notebook has constructed all four supported LNG routes as independent NeqSim models,
calculated common performance metrics, exposed their equipment manifests and reusable product
streams, checked balances and temperature approaches, compared specific energy with four
published points, and tested SMR exchanger-grid sensitivity.

Suggested extensions:

1. optimize each refrigerant composition and circulation rate on the same constraints;
2. add measured pressure drops and compressor or expander maps;
3. compare SRK with Peng–Robinson and a calibrated model;
4. connect a dehydration and heavy-hydrocarbon-removal process upstream;
5. route flash gas to fuel or recompression and include its energy credit consistently; and
6. perform uncertainty analysis on ambient temperature, feed richness, driver efficiency,
   pressure drop, and exchanger approach.

**NeqSim references**

- [LNG liquefaction process documentation](https://equinor.github.io/neqsim/process/lng_liquefaction.html)
- [LNGProcessBuilder source](https://github.com/equinor/neqsim/blob/master/src/main/java/neqsim/process/lng/LNGProcessBuilder.java)
- [Process simulation guide](https://equinor.github.io/neqsim/process/process_simulation.html)


In [ ]:
machine_readable_results = {
    "neqsim_build": NEQSIM_BUILD,
    "neqsim_source_commit": REQUIRED_NEQSIM_COMMIT,
    "basis": {
        "feed_flow_kg_per_h": FEED_FLOW_KG_PER_H,
        "feed_temperature_c": FEED_TEMPERATURE_C,
        "feed_pressure_bara": FEED_PRESSURE_BARA,
        "product_pressure_bara": PRODUCT_PRESSURE_BARA,
        "target_liquefaction_temperature_c": (
            TARGET_LIQUEFACTION_TEMPERATURE_C
        ),
        "minimum_approach_c": MINIMUM_APPROACH_C,
        "initial_zones": NUMBER_OF_ZONES,
        "compressor_isentropic_efficiency": COMPRESSOR_EFFICIENCY,
        "expander_isentropic_efficiency": EXPANDER_EFFICIENCY,
    },
    "literature_specific_energy_kwh_per_kg_lng": (
        literature_table[
            "reference specific energy [kWh/kg LNG]"
        ].to_dict()
    ),
    "route_results": (
        comparison_table.reset_index()
        .replace({np.nan: None})
        .to_dict(orient="records")
    ),
    "smr_zone_sensitivity": (
        zone_sensitivity_table
        .replace({np.nan: None})
        .to_dict(orient="records")
    ),
    "checks_passed": len(verification_checks),
}

results_json = json.dumps(
    machine_readable_results,
    indent=2,
    sort_keys=True,
)
print(results_json)
